In [ ]:
%cd ..

# Features Extraction

In [ ]:
from dataclasses import dataclass
import pickle
import numpy as np
import mdtraj as md
from maniprot.core.helpers import lcs, lcs_args
from maniprot.core.SO3 import SO3Manifold
from maniprot.core.pointcloud import PointcloudManifold

@dataclass(slots=True)
class NBConfig:
    # Features to compute
    compute_orientation: bool = True
    compute_orientation_IM: bool = True
    compute_orientation_IM_u: bool = True
    compute_orientation_IM_theta: bool = True
    compute_pointcloud: bool = True
    compute_pointcloud_IM: bool = True
    compute_ca_coordinates: bool = True
    compute_torsion_angles: bool = True

    # Feature extraction options
    stride: int = 1
    force_superposition: bool = True
    demean: bool = True
    normalize: bool = True

    # Orientation parameters
    orientation_learning_rate: float = 0.1
    orientation_threshold: float = 0.001
    orientation_max_steps: int = 256

    # Pointcloud parameters
    pointcloud_delta: float = 0.1
    pointcloud_learning_rate: float = 1
    pointcloud_threshold: float = 1
    pointcloud_max_steps: int = 256

    # Parallelization
    n_jobs: int = -1  # -1 for all cores

    @property
    def features_to_demean(self) -> list[str]:
        if not self.demean:
            return []

        features = []
        if self.compute_ca_coordinates:
            features.append("Cα coordinates")
        if self.compute_torsion_angles:
            features.append("Torsion angles")

        return features

    @property
    def features_to_normalize(self) -> list[str]:
        if not self.normalize:
            return []

        features = []
        if self.compute_ca_coordinates:
            features.append("Cα coordinates")
        if self.compute_torsion_angles:
            features.append("Torsion angles")

        return features

In [ ]:
# Warmup numba functions
def _warmup_numba():
    """Compile numba functions."""

    # Warmup SO3 Manifold
    dummy_theta = np.array([np.pi // i for i in range(1, 11)])
    dummy_lcs = np.ascontiguousarray(np.swapaxes(np.array([
        [np.cos(dummy_theta), np.sin(dummy_theta), np.zeros_like(dummy_theta)],
        [-np.sin(dummy_theta), np.cos(dummy_theta), np.zeros_like(dummy_theta)],
        [np.zeros_like(dummy_theta), np.zeros_like(dummy_theta), np.ones_like(dummy_theta)]
    ]), -1, 0))

    manifold = SO3Manifold().fit(
        dummy_lcs[None], dummy_lcs,
        threshold=float("-inf"), max_steps=2
    )
    manifold.transform(dummy_lcs[None])

    del dummy_theta, dummy_lcs, manifold


    # Warmup Pointcloud Manifold
    dummy_pointcloud = np.random.randn(2, 5, 3)

    manifold = PointcloudManifold(delta=0.1).fit(
        dummy_pointcloud, dummy_pointcloud[0],
        threshold=float("-inf"), max_steps=2
    )

    del dummy_pointcloud, manifold

_warmup_numba()

In [ ]:
def _compute_orientation_features(
    trajectory: md.Trajectory,
    reference: md.Trajectory,
    learning_rate: float,
    threshold: float,
    max_steps: int,
) -> np.ndarray:
    # Get local coordinate systems
    trajectory_lcs = lcs(*lcs_args(trajectory))
    reference_lcs = lcs(*lcs_args(reference))[0]

    # Fit manifold
    manifold = SO3Manifold().fit(
        trajectory_lcs, reference_lcs,
        learning_rate=learning_rate,
        threshold=threshold,
        max_steps=max_steps
    )

    # Get intrinsic coordinates
    features = manifold.transform(trajectory_lcs)
    return features

def _compute_pointcloud_features(
    trajectory: md.Trajectory,
    reference: md.Trajectory,
    delta: float,
    learning_rate: float,
    threshold: float,
    max_steps: int,
) -> np.ndarray:
    # Get point cloud coordinates
    topology: md.Topology = trajectory.top # type: ignore
    ca_indices = topology.select("protein and backbone and name CA")
    trajectory_pointcloud: np.ndarray = trajectory.xyz[:, ca_indices] # type: ignore
    reference_pointcloud: np.ndarray = reference.xyz[0, ca_indices] # type: ignore

    # Fit manifold
    manifold = PointcloudManifold(delta=delta).fit(
        trajectory_pointcloud, reference_pointcloud,
        learning_rate=learning_rate,
        threshold=threshold,
        max_steps=max_steps
    )

    # Get intrinsic coordinates
    features = manifold.transform(trajectory_pointcloud)
    return features

def _compute_ca_coordinates_features(trajectory: md.Trajectory, reference: md.Trajectory) -> np.ndarray:
    topology: md.Topology = trajectory.top # type: ignore
    ca_indices = topology.select("protein and backbone and name CA")

    features = trajectory.xyz[:, ca_indices] # type: ignore
    return features

def _compute_torsion_angles_features(trajectory: md.Trajectory, reference: md.Trajectory) -> np.ndarray:
    phi = md.compute_phi(trajectory)[1]
    psi = md.compute_psi(trajectory)[1]

    # Add padding for N-terminal (phi) and C-terminal residues (psi)
    traj_phi = np.concatenate([np.zeros((len(trajectory), 1)), phi], axis=-1)
    traj_psi = np.concatenate([psi, np.zeros((len(trajectory), 1))], axis=-1)

    # Combine phi and psi angles
    torsion_angles = np.concatenate([traj_phi, traj_psi], axis=-1)

    # Convert torsion angles to sine and cosine representation
    torsion_sin = np.sin(torsion_angles)
    torsion_cos = np.cos(torsion_angles)

    # Combine sine and cosine features
    features = np.concatenate([torsion_cos, torsion_sin], axis=-1)
    return features

def compute_features(trajectory: md.Trajectory, reference: md.Trajectory, config: NBConfig) -> dict[str, np.ndarray]:
    features = {}

    if config.compute_orientation:

        # max steps 0 for no iterations and projection at the reference
        features["Orientation"] = _compute_orientation_features(
            trajectory, reference,
            config.orientation_learning_rate,
            config.orientation_threshold,
            0,
        )

    if config.compute_orientation_IM:

        features["Orientation ☉"] = _compute_orientation_features(
            trajectory, reference,
            config.orientation_learning_rate,
            config.orientation_threshold,
            config.orientation_max_steps,
        )

    if config.compute_orientation_IM_u:

        orientation_IM = features.get("Orientation ☉")

        if orientation_IM is None:
            raise ValueError("Orientation ☉ must be enabled to compute its variants")

        features["Orientation ☉ (u)"] = orientation_IM / np.linalg.norm(orientation_IM, axis=-1, keepdims=True) 

    if config.compute_orientation_IM_theta:

        orientation_IM = features.get("Orientation ☉")

        if orientation_IM is None:
            raise ValueError("Orientation ☉ must be enabled to compute its variants")

        features["Orientation ☉ (θ)"] = np.linalg.norm(orientation_IM, axis=-1)

    if config.compute_pointcloud:

        # max steps 0 for no iterations and projection at the reference
        features["Pointcloud"] = _compute_pointcloud_features(
            trajectory, reference,
            config.pointcloud_delta,
            config.pointcloud_learning_rate,
            config.pointcloud_threshold,
            0,
        )

    if config.compute_pointcloud_IM:

        features["Pointcloud ☉"] = _compute_pointcloud_features(
            trajectory, reference,
            config.pointcloud_delta,
            config.pointcloud_learning_rate,
            config.pointcloud_threshold,
            config.pointcloud_max_steps,
        )

    if config.compute_ca_coordinates:
        features["Cα coordinates"] = _compute_ca_coordinates_features(trajectory, reference)

    if config.compute_torsion_angles:
        features["Torsion angles"] = _compute_torsion_angles_features(trajectory, reference)


    if config.demean:
        for key in config.features_to_demean:
            features[key] -= np.mean(features[key], axis=0, keepdims=True)

    if config.normalize:
        for key in config.features_to_normalize:
            std_dev = np.std(features[key], axis=0, keepdims=True)
            std_dev[std_dev == 0] = 1.0
            features[key] /= std_dev


    return features

In [ ]:
top_pdb = "examples/_structure.pdb"
traj_dcd = "examples/_trajectory.dcd"

config = NBConfig()

In [ ]:
# Load trajectory
traj = md.load(traj_dcd, top=top_pdb)

# Load reference as trajectory
ref = md.load(top_pdb)

# Center & align
ref.center_coordinates()
traj.center_coordinates()
traj.superpose(ref, 0)

In [ ]:
features = compute_features(traj, ref, config)
with open("examples/features.pkl", "wb") as h:
    pickle.dump({
        "data": features
    }, h, protocol=pickle.HIGHEST_PROTOCOL)